In [61]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


In [62]:
#Configuração inicial dos paramentros
np.random.seed(42)

# Número de dias úteis por ano
dias_por_ano = 252
anos = [2020, 2021, 2022, 2023, 2024]
total_dias = dias_por_ano * len(anos)

# Média anual e volatilidade anual aproximada para cada ativo
params = {
    'IBOV': {'mu': 0.08, 'sigma': 0.25},
    'SP500': {'mu': 0.10, 'sigma': 0.20},
    'USDBRL': {'mu': 0.02, 'sigma': 0.05},
    'OURO': {'mu': 0.06, 'sigma': 0.15},
    'VIX': {'mu': 0.20, 'sigma': 0.10},
    'IPCA': {'mu': 0.04, 'sigma': 0.015}
}

# Criar datas diárias
dates = pd.bdate_range(start='2020-01-02', periods=total_dias)


In [63]:
#geração de dados
df = pd.DataFrame(index=dates)

for ativo in ['IBOV', 'SP500', 'USDBRL', 'OURO']:
    # Converter média e volatilidade anual para diária
    mu_d = params[ativo]['mu'] / dias_por_ano
    sigma_d = params[ativo]['sigma'] / np.sqrt(dias_por_ano)
    
    # Simular log-retornos diários
    ret_diario = np.random.normal(mu_d, sigma_d, total_dias)
    
    # Calcular preços a partir de um valor inicial arbitrário
    preco_inicial = 100
    df[ativo] = preco_inicial * np.exp(np.cumsum(ret_diario))
    df[f'ret_log_{ativo}'] = ret_diario

# VIX e IPCA diários
df['VIX'] = np.random.normal(params['VIX']['mu'], params['VIX']['sigma'], total_dias)
df['IPCA'] = np.random.normal(params['IPCA']['mu']/dias_por_ano, params['IPCA']['sigma']/np.sqrt(dias_por_ano), total_dias)


In [64]:
#Dados para serem usados no modelo
ativos = ['IBOV', 'SP500', 'USDBRL', 'OURO']
lags = 1
features = []

for ativo in ativos:
    df[f'{ativo}_lag1'] = df[f'ret_log_{ativo}'].shift(lags)
    features.append(f'{ativo}_lag1')

df.dropna(inplace=True)

X = df[features]
y = df[[f'ret_log_{ativo}' for ativo in ativos]]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

rf_models = {}
y_pred = pd.DataFrame(index=y_test.index)

for ativo in ativos:
    rf = RandomForestRegressor(n_estimators=200, random_state=42)
    rf.fit(X_train, y_train[f'ret_log_{ativo}'])
    rf_models[ativo] = rf
    y_pred[ativo] = rf.predict(X_test)


In [65]:
#Alocação da carteira
alloc = {
    'IBOV': 0.25,
    'SP500': 0.35,
    'USDBRL': 0.20,
    'OURO': 0.20
}

df['ret_carteira'] = sum(df[f'ret_log_{ativo}'] * alloc[ativo] for ativo in ativos)


In [66]:
#Calculo do risco
df['Ano'] = df.index.year
risco_ano = df.groupby('Ano')[['ret_carteira']].std() * np.sqrt(dias_por_ano) * 100
print(tabela_final.columns)
print(len(tabela_final.columns))


Index(['IBOV', 'SP500', 'USDBRL', 'OURO', 'Carteira', 'Risco_Carteira', 'VIX',
       'IPCA'],
      dtype='object')
8


In [67]:
# Função para converter retorno logarítmico para percentual
def ret_to_percent(log_ret):
    return (np.exp(log_ret) - 1) * 100

# Ativos
ativos = ['IBOV', 'SP500', 'USDBRL', 'OURO']

# Retorno anual
ret_ano = df.groupby('Ano')[[f'ret_log_{ativo}' for ativo in ativos] + ['ret_carteira']].sum()
ret_ano = ret_ano.apply(ret_to_percent)

# Risco anual (volatilidade da carteira)
dias_por_ano = 252
risco_ano = df.groupby('Ano')[['ret_carteira']].std() * np.sqrt(dias_por_ano) * 100

# VIX e IPCA anual
vix_ano = df.groupby('Ano')['VIX'].mean() * 100
ipca_ano = df.groupby('Ano')['IPCA'].sum() * 100

# Concatenar tudo
tabela_final = pd.concat([ret_ano, risco_ano, vix_ano, ipca_ano], axis=1)

# Renomear colunas corretamente (8 colunas)
tabela_final.columns = [*ativos, 'Carteira', 'Risco_Carteira', 'VIX', 'IPCA']

# Mostrar tabela final
tabela_final



,IBOV,SP500,USDBRL,OURO,Carteira,Risco_Carteira,VIX,IPCA
Ano,,,,,,,,
2020,9.559862,55.904982,9.696339,-18.473996,16.872282,9.810088,20.206704,2.451874
2021,7.769157,32.477170,-4.759444,-11.581802,8.628980,9.840258,19.010571,2.974830
2022,-6.278660,1.785871,0.521740,-6.356813,-2.186797,9.834303,19.517419,4.766127
2023,106.560359,2.734591,5.330739,-5.061121,21.021473,9.207838,19.731493,3.552579
2024,37.927100,17.612410,3.209424,19.663700,19.648528,9.767164,19.712911,3.296435


In [68]:


plt.figure(figsize=(12,6))
sns.lineplot(data=tabela_final[['Carteira', 'IBOV', 'SP500', 'USDBRL', 'OURO']], marker="o")
plt.title("Retorno Anual (%) - Carteira e Ativos (Simulação)", fontsize=14)
plt.ylabel("Retorno (%)", fontsize=12)
plt.xlabel("Ano", fontsize=12)
plt.xticks(rotation=45)
plt.legend(title="Ativos")
plt.tight_layout()
plt.show()


NameError: name 'sns' is not defined

<Figure size 1200x600 with 0 Axes>